## Load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("C:/Users/hp/predictive-maintenance-diagnosis/data/processed/mafaulda_features_final.csv")
print(df.shape)
df.head()
df.info()

(244000, 74)


,tachometer_max,tachometer_min,tachometer_mean,tachometer_sd,tachometer_rms,tachometer_skewness,tachometer_kurtosis,tachometer_crest,tachometer_form,underhang_axial_max,...,microphone_min,microphone_mean,microphone_sd,microphone_rms,microphone_skewness,microphone_kurtosis,microphone_crest,microphone_form,fault_type,file_id
0,4.8255,-1.03430,0.285707,1.923731,1.944832,1.746167,1.069547,2.481191,1.549306,1.4481,...,-0.21188,0.021197,0.125715,0.127490,0.613978,-0.122630,3.971303,1.228815,normal,C:\Users\hp\predictive-maintenance-diagnosis\d...
1,5.0222,-0.90783,-0.354661,0.913615,0.980039,5.163782,25.074871,5.124488,1.498203,1.4726,...,-0.16331,-0.005893,0.093443,0.093629,0.571778,-0.491293,3.312444,1.186456,normal,C:\Users\hp\predictive-maintenance-diagnosis\d...
2,4.6159,-1.00020,0.347854,1.976150,2.006532,1.640818,0.711634,2.300436,1.543382,1.4064,...,-0.23216,0.022073,0.127320,0.129219,0.578973,-0.244876,3.809893,1.223066,normal,C:\Users\hp\predictive-maintenance-diagnosis\d...
3,5.0479,-0.96772,-0.424213,0.731489,0.845596,6.560812,42.284708,5.969635,1.384001,1.4884,...,-0.17156,-0.008245,0.089225,0.089605,0.539192,-0.477678,3.378053,1.188460,normal,C:\Users\hp\predictive-maintenance-diagnosis\d...
4,4.7578,-0.98375,0.428474,2.041294,2.085778,1.512567,0.305607,2.281067,1.530529,1.4641,...,-0.21424,0.021832,0.124984,0.126877,0.594333,-0.240726,3.597660,1.230027,normal,C:\Users\hp\predictive-maintenance-diagnosis\d...


## Data quality 

In [4]:
print("Nulls:", df.isnull().sum().sum())
print("Duplicates:", df.duplicated().sum())

df = df.drop_duplicates().reset_index(drop=True)
print("Shape after dedup:", df.shape)

# check for constant/near-zero-variance features
feature_cols = [c for c in df.columns if c != 'fault_type']
print("\nFeatures with near-zero variance:")
print(df[feature_cols].std().sort_values().head(5))

Nulls: 0
Duplicates: 0
Shape after dedup: (244000, 74)

Features with near-zero variance:


TypeError: could not convert string to float: 'C:\\Users\\hp\\predictive-maintenance-diagnosis\\data\\raw\\MAFAULDA\\normal\\12.288.csv'

=> No missing values and no duplicate rows were found — the dataset is clean at the row level.

## Class balance

In [ ]:
print(df['fault_type'].value_counts())
print(df['fault_type'].value_counts(normalize=True).round(3))

df['fault_type'].value_counts().plot(kind='bar', figsize=(7,4))
plt.title('Fault type distribution')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

=> The dataset remains imbalanced: underhang_bearing (28.6%) and overhang_bearing (26.3%) together make up more than half of all samples, while normal represents only 2.5% (5,978 samples).

=> This is a meaningful imbalance and will be addressed during model training through stratified splitting and class-weighted loss, so that the model doesn't default to favoring the majority. 

In [ ]:
key_features = ['underhang_axial_kurtosis', 'overhang_radial_rms', 
                 'microphone_kurtosis', 'tachometer_mean']

df[key_features].hist(figsize=(12,8), bins=50)
plt.tight_layout()
plt.show()

##  Feature/label split

In [ ]:
X = df[feature_cols]
y = df['fault_type']
print(X.shape, y.shape)

## Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## Correlation heatmap

In [ ]:
corr = X.corr()

plt.figure(figsize=(14,12))
plt.imshow(corr)
plt.colorbar()
plt.title('Feature correlation heatmap')
plt.tight_layout()
plt.show()

high_corr = corr.where((abs(corr) > 0.9) & (corr != 1.0)).stack()
print(f"Highly correlated pairs (>0.9): {len(high_corr)//2}")
print(high_corr.head(20))

=> Correlation is concentrated within individual sensor channels, which is expected since multiple statistics computed from the same signal naturally move together. Cross-channel correlation is generally weak (near 0), meaning different sensors are capturing largely independent information, supporting the value of using all 8 channels rather than redundant data.

## PCA separability

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
proj = pca.fit_transform(X_scaled)

plt.figure(figsize=(8,6))
for label in y.unique():
    mask = (df['fault_type'] == label).values
    plt.scatter(proj[mask,0], proj[mask,1], label=label, alpha=0.3, s=4)
plt.legend(markerscale=3)
plt.title('PCA — fault type separability')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')
plt.tight_layout()
plt.show()

=> PCA shows underhang and overhang bearing faults as visually distinct from other conditions, but normal, imbalance, and misalignment classes overlap heavily near the origin. Since PC1+PC2 only explain ~32% of total variance, this is likely a limitation of linear 2D projection.

## t-SNE 

In [ ]:
from sklearn.manifold import TSNE

sample_df = df.sample(n=5000, random_state=42)
sample_idx = sample_df.index
X_sample = X_scaled[sample_idx]
y_sample = sample_df['fault_type']

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
proj_tsne = tsne.fit_transform(X_sample)

plt.figure(figsize=(8,6))
for label in y_sample.unique():
    mask = (y_sample == label).values
    plt.scatter(proj_tsne[mask,0], proj_tsne[mask,1], label=label, alpha=0.5, s=8)
plt.legend(markerscale=3)
plt.title('t-SNE — fault type separability (5000-sample subset)')
plt.tight_layout()
plt.show()

=> t-SNE reveals substantially clearer class separation than PCA, with underhang and overhang bearing faults forming multiple distinct clusters, and imbalance/vertical misalignment grouping together in a separate region. This confirms the apparent overlap in PCA was a limitation of linear projection, not a lack of real separability. The classes are genuinely distinguishable in the full feature space, which supports feasibility for the diagnostic classifier.

## Outlier check

In [ ]:
Q1 = X.quantile(0.25)
Q3 = X.quantile(0.75)
IQR = Q3 - Q1
outlier_counts = ((X < (Q1 - 1.5*IQR)) | (X > (Q3 + 1.5*IQR))).sum()
print(outlier_counts.sort_values(ascending=False).head(15))

In [ ]:
df.to_csv('predictive-maintenance-diagnosis/data/processed/mafaulda_features_final.csv', index=False)
print("Saved:", df.shape) 

In [ ]:
# Check outlier counts split by fault_type, for the features that showed the most outliers globally
top_outlier_features = ['overhang_radial_mean', 'underhang_tangential_mean', 'overhang_radial_min',
                         'underhang_radial_mean', 'overhang_radial_rms']

for feature in top_outlier_features:
    print(f"\n--- {feature} ---")
    for fault in df['fault_type'].unique():
        subset = df[df['fault_type'] == fault][feature]
        Q1, Q3 = subset.quantile([0.25, 0.75])
        # using the GLOBAL IQR bounds (from the whole dataset), not per-class,
        # to see how many of each class's points fall outside the global "normal" range
        global_Q1, global_Q3 = df[feature].quantile([0.25, 0.75])
        global_IQR = global_Q3 - global_Q1
        lower = global_Q1 - 1.5*global_IQR
        upper = global_Q3 + 1.5*global_IQR
        
        n_outliers = ((subset < lower) | (subset > upper)).sum()
        pct = n_outliers / len(subset) * 100
        print(f"  {fault}: {n_outliers} outliers ({pct:.1f}% of this class's samples)")